In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "wolf2020human")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Data for Wolf & Tomasello 2020 - Study 2.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)

df['study_id']="wolf2020human"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns

In [3]:
df.rename(columns={"subjectname": "ape", 
    "movietime":"movie_time",
    "aftertime":"after_time",
    "species":"species_original",
    "rearinghistory":"rearing_history",
    "session1timeseconds":"1_timeseconds",
    "session2timeseconds":"2_timeseconds"}, inplace=True)
# df['condition_1']='sharing'
# df['session_1']='1'
# df['condition_2']='observing'
# df['session_2']='2'

common_ground = df[df.order.str.contains("common ground first")]
c_list_1 =common_ground[['study_id','ape','age','order','movie_time']].copy()
c_list_1['session_1']='1'
c_list_1['condition_1']='sharing'
c_list_2 =common_ground[['study_id','ape','age','order','after_time']].copy()
c_list_2['session_2']='2'
c_list_2['condition_2']='observing'

observation = df[df.order.str.contains("observation first")]
o_list_1 =observation[['study_id','ape','age','order','after_time']].copy()
o_list_1['session_1']='1'
o_list_1['condition_1']='observing'
o_list_2 =observation[['study_id','ape','age','order','movie_time']].copy()
o_list_2['session_2']='2'
o_list_2['condition_2']='sharing'
# df.columns

In [4]:
df_temp1 = c_list_1.values.tolist() + c_list_2.values.tolist() + o_list_1.values.tolist() + o_list_2.values.tolist() 

df = pd.DataFrame(df_temp1, columns=['study_id','ape','age','order','approach_latency', 'session', 'condition'])



# df = df.melt(id_vars=["study_id","ape", "movie_time", "age", "order", "after_time","rearing_history",],
#                           value_vars=["1", "2"],
#                           var_name="session", value_name="time_in_seconds")


# df = df.loc[:,~df.columns.duplicated()]

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [6]:
# df.columns
df.rename(columns={"ape": "participant",   
                    "age":"age_in_years"}, inplace=True)

order_update = [['common ground first','sharing_first'],
                ['observation first','observing_first']]
for x,y in order_update:
    df['order'].replace(x, y, inplace=True, regex=True)



In [7]:
wolf2020human_standardized=df[['study_id','participant', 'age_in_years','sex','species', 'session','condition', 'order',
            'approach_latency']]

wolf2020human_standardized=wolf2020human_standardized.sort_values(by = ['participant','session'])
comp_out_path_stand = os.path.join(out_pathway, 'wolf2020human_exp2_standardized.csv')
wolf2020human_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =wolf2020human_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
wolf2020human_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'wolf2020human_exp2_glossary.csv')
wolf2020human_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)